# Prepare PFTNC inference input

The PFTNC inference pipeline performs the feature-engineering step itself. This notebook only selects the required source rows from the feature-engineering output and writes them as a CSV file for the EOAP.

Update the input path, output path, year. The inference package currently expects the date/time column to be timezone-naive, so the preparation step removes timezone information before writing the CSV.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
input_path = Path("path/to/source/dataset.csv")
output_path = Path("input/<your-site>/inference_input.csv")
year = 2023
date_column = "time"

required_columns = ['site',
 'date',
 'time',
 'site_lat',
 'site_lon',
 'chime_diatoms_ug_per_l',
 'chime_cyanobacteria_ug_per_l',
 'chime_others_ug_per_l',
 'lstm_lswt_c',
 's2_chl_ug_per_l',
 's2_tur_mg_per_l',
 's2_cdom_1_per_m',
 's3_chl_ug_per_l',
 's3_tur_mg_per_l',
 's3_cdom_1_per_m',
 'diatoms_ug_per_l',
 'cyanobacterial_ug_per_l',
 'others_ug_per_l']

In [ ]:
df = pd.read_csv(input_path)
df.head()

In [ ]:
df.tail()

In [ ]:
# Normalize timestamps to UTC, then remove timezone information.
# This avoids a tz-aware/tz-naive comparison error in PFTNC inference.
df[date_column] = pd.to_datetime(df[date_column], utc=True).dt.tz_localize(None)
df_year = df[df[date_column].dt.year == year].copy()

if required_columns is not None:
    missing_columns = sorted(set(required_columns) - set(df_year.columns))
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")
    df_year = df_year[required_columns]

df_year.shape

In [ ]:
df_year

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
df_year.to_csv(output_path, index=False)
print(f"Saved {len(df_year)} rows to {output_path}")